# speech2face on RunPod — PersonaPlex **base7b RAG** + IMTalker real-time talking avatar

Full-duplex speech-to-speech with **retrieval-augmented generation** driving a lip-synced
talking head in the browser. You talk; the **base7b RAG fine-tune** (the full 32-layer
PersonaPlex 7B backbone) listens, transcribes your question (STT), retrieves a passage (local
dense index + web search), injects a 16-token fact card into its own pause, and speaks a
**grounded answer** in the Robert_5 voice while the face lip-syncs. Full duplex: you can talk
over it (barge-in) and it hears you — there is no turn-taking state machine.

This notebook runs the **RAG configuration** of the speech2face server on **any x86_64 NVIDIA
GPU** (A100, H100, L40S, A6000 …):

| part | what |
|---|---|
| speech engine | PersonaPlex **base7b RAG fine-tune** (`sadmN/moshi-base7b-rag`, `ckpt/rag_base7b/step_1500`), served through `src/moshi_local/moshi_distill` with the single-path streaming protocol — the same model, loader and serving protocol as the standalone `server_rag_stream.py` server |
| RAG | STT (`kyutai/stt-1b-en_fr-candle`) runs in parallel (transcript + VAD, non-driving); dense MiniLM index + cross-encoder rerank (local) and Tavily web search (hybrid) fire at the user's mid-pause; 16-token fact card in the model's own pause on `<lookup>` |
| avatar | causal motion student + distilled renderer (the unchanged try19 pipeline), driven by the model's own transformer hidden states |
| legacy | the full-PersonaPlex 7B configuration is still here: `./run_x86.sh` (see README_RUNPOD.md) |

### Before you start

| | |
|---|---|
| **GPU** | compute capability ≥ 8.0 (Ampere or newer), **≥ 32 GB VRAM** (the bf16 7B RAG model is 16 GB; with Mimi + STT + retrievers + avatar ≈ 22–25 GB — a 24 GB card is too tight) |
| **Disk** | **≥ 55 GB** on `/workspace` (base7b student ~16 GB + STT/retrievers ~3 GB + avatar weights ~3 GB + Python env ~9 GB + HF cache; the 16 GB full-7B base is skipped by default — see `SKIP_BASE_LM` in the config cell) |
| **Pod template** | a RunPod *PyTorch* or *CUDA 12* image is fine — the notebook builds its own Python 3.10 env, so the base image's Python does not matter |
| **Exposed port** | add **8991** to the pod's *HTTP ports* if you want the RunPod proxy URL. Otherwise use the Cloudflare tunnel cell — it needs no pod configuration |
| **HF token** | a Hugging Face token whose account has **accepted the licence of `nvidia/personaplex-7b-v1`** (gated — accept it on the model page first, or the weight download fails) |

### What the cells do

1. check the GPU, driver and disk
2. install the handful of system libraries the server needs
3. clone the repo into `/workspace/speech2face` and validate the RAG integration files
4. write the x86_64 build scripts into the clone
5. `setup_env_x86.sh` — Python 3.10 env in `./env` (~10–25 min)
6. `prepare_x86.sh` — ~23 GB of weights (incl. the base7b student + STT + retrievers), GPU probe, TensorRT engine (~20–50 min)
7. start the **RAG server**, wait for it to load, print an **HTTPS** URL (the microphone needs HTTPS)
8. headless end-to-end check with `tools/test_session.py`

Total first run: roughly 45–80 minutes, almost all of it downloading. Everything lands on
`/workspace`, so a restarted pod with the same volume skips steps 5 and 6.


## 1 · Configuration

The only thing you normally change here is the Hugging Face token. `RETRIEVER` selects the
passage source: `hybrid` (web search first, local index as fallback — the default), `local`
(fully offline) or `web` (drops the local index models, saves ~1 GB of VRAM).


In [ ]:
import os, subprocess, sys, time, json, shutil, urllib.request, getpass

# ----------------------------------------------------------------- settings
REPO_URL  = "https://github.com/MoshiHead/speech2face_orin_n_original_add_s_student_rag_model_moshi-base7b-rag-.git"
REPO_DIR  = "/workspace/speech2face"      # the clone lands here (folder name: speech2face)
BRANCH    = ""                            # "" = the repo's default branch
PORT      = 8991                          # 8998 is refused by the server
VOICE     = "Robert_5.pt"                 # the avatar's voice (motion student was trained on it)
TEXT_PROMPT = ("You are Robert, an assistant from RB Labs. You have knowledge about crypto, "
               "finance, investment, technology, and general topics. Answer every question "
               "completely, descriptively, and with useful detail.")
RETRIEVER = "hybrid"                      # hybrid | local | web

# Disk saver for the RAG config: skip the 16 GB full-7B model.safetensors from the base repo
# (only the legacy run_x86.sh config needs it; the base7b RAG model does not). This brings the
# download from ~38 GB to ~23 GB. Set to "0" if you also want the legacy full-PersonaPlex config.
os.environ["SKIP_BASE_LM"] = "1"

# Hugging Face token, in order of preference:
#   1. the HF_TOKEN environment variable (set it in the pod's env vars - best on a shared pod)
#   2. /workspace/hf_token.txt
#   3. a prompt, below
# Pasting it straight into this cell also works, but then do not commit the notebook.
token = "GjsPkoyTVNNHDFJNYQCBGQGdYUdKgadHsI"
HF_TOKEN = "hf_" + token
if not HF_TOKEN and os.path.exists("/workspace/hf_token.txt"):
    HF_TOKEN = open("/workspace/hf_token.txt").read().strip()
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face token (hf_...): ").strip()
assert HF_TOKEN.startswith("hf_"), "that does not look like a Hugging Face token"
os.environ["HF_TOKEN"] = HF_TOKEN

# ----------------------------------------------------------------- helper
def sh(cmd, cwd=None, check=True, env=None, quiet=False):
    "Run a shell command, streaming its output into the notebook as it happens."
    cwd = cwd or (REPO_DIR if os.path.isdir(REPO_DIR) else os.getcwd())
    if not quiet:
        print("$ " + cmd, flush=True)
    p = subprocess.Popen(["bash", "-c", cmd], cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1,
                         errors="replace", env={**os.environ, **(env or {})})
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError("command failed (exit %d): %s" % (rc, cmd))
    return rc

print("repo     :", REPO_URL)
print("into     :", REPO_DIR)
print("port     :", PORT)
print("retriever:", RETRIEVER)
print("token    : %s...%s  (%d chars)" % (HF_TOKEN[:6], HF_TOKEN[-4:], len(HF_TOKEN)))


In [ ]:
!pip install huggingface_hub


## 2 · Check the GPU, driver and disk

A compute capability below 8.0, a driver older than 525, or less than ~75 GB free on `/workspace` will stop you later — better to see it now.


In [ ]:
sh("nvidia-smi --query-gpu=index,name,compute_cap,memory.total,driver_version "
   "--format=csv || echo 'NO GPU VISIBLE'", cwd="/")
sh("echo; df -h /workspace / 2>/dev/null | sed -n '1,5p'; echo; uname -m", cwd="/")

# a quick sanity read of the numbers above
out = subprocess.run(["bash", "-c", "nvidia-smi --query-gpu=compute_cap,memory.total "
                      "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
if out.strip():
    cap, mem = [x.strip() for x in out.strip().splitlines()[0].split(",")]
    gib = float(mem) / 1024
    print("\ncompute capability %s, %.0f GiB of VRAM" % (cap, gib))
    if float(cap) < 8.0:
        print("!! this pipeline needs Ampere or newer (compute capability >= 8.0)")
    elif gib < 22:
        print("!! under 22 GiB of VRAM - the RAG stack (student + STT + retrievers + avatar) needs ~16-18 GiB")
    else:
        print("ok")


## 3 · System libraries

`libsndfile1` for soundfile, `ffmpeg` for torchaudio/librosa, `libgl1` + `libglib2.0-0` for OpenCV, `util-linux` for the watchdog's `flock`. Already present in most RunPod images, in which case this is a no-op.


In [ ]:
sh("export DEBIAN_FRONTEND=noninteractive && "
   "apt-get update -qq && "
   "apt-get install -y -qq --no-install-recommends "
   "  git curl ca-certificates bzip2 unzip util-linux "
   "  ffmpeg libsndfile1 libgl1 libglib2.0-0 "
   "&& echo '--- installed ---' && ffmpeg -version | head -1", cwd="/")


## 4 · Clone the repo

Re-running this cell on an existing clone just fetches and fast-forwards it; the weights you already downloaded into `weights/`, `assets/`, `rag/` and `hf/` are untouched.


In [ ]:
os.makedirs(os.path.dirname(REPO_DIR), exist_ok=True)
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(REPO_DIR, "already exists - updating")
    # FETCH_HEAD, not origin/HEAD: a --depth 1 clone has no remote HEAD ref
    sh("git -C %s fetch --depth 1 origin %s && git -C %s reset --hard FETCH_HEAD"
       % (REPO_DIR, BRANCH or "HEAD", REPO_DIR), cwd="/", check=False)
else:
    branch = ("--branch " + BRANCH) if BRANCH else ""
    sh("git clone --depth 1 %s %s %s" % (branch, REPO_URL, REPO_DIR), cwd="/")

# every later cell (including the %%writefile ones) is relative to the clone
os.chdir(REPO_DIR)
print("\ncwd:", os.getcwd())
sh("ls -la", check=False)


## 4b · Validate the RAG integration files

The integrated system needs files that only exist in the **updated** repo: the RAG engine
(`src/niloys/live/rag_student_engine.py`), its wrapper (`imtalker_rag_student_live.py`), the
`src/moshi_local/` model package and the `rag/` assets (tokenizer + local index). If the model
package or the assets are missing (stale clone), they are restored from the
`sadmN/moshi-base7b-rag` zip; the two Python modules must come from the repo.


In [ ]:
import zipfile
from huggingface_hub import hf_hub_download

# the model package + RAG assets can be restored from the HF zip if absent
need_zip = (not os.path.exists("src/moshi_local/moshi_distill/models/lm.py")
            or not os.path.exists("rag/rag_index_dense/passages.jsonl")
            or not os.path.exists("rag/tokenizer.model"))
if need_zip:
    print("src/moshi_local or rag/ incomplete - restoring from sadmN/moshi-base7b-rag ...")
    z = hf_hub_download("sadmN/moshi-base7b-rag", "moshi_base7b_server.zip", token=HF_TOKEN)
    tmp = "/tmp/_base7b_zip"
    shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(tmp)
    base = tmp
    for sub, _, _ in os.walk(tmp):
        if os.path.isdir(os.path.join(sub, "moshi_local", "moshi_distill")):
            base = sub
            break
    os.makedirs("src", exist_ok=True)
    os.makedirs("rag", exist_ok=True)
    if os.path.isdir(os.path.join(base, "moshi_local")) \
            and not os.path.exists("src/moshi_local/moshi_distill/models/lm.py"):
        shutil.copytree(os.path.join(base, "moshi_local"), "src/moshi_local",
                        dirs_exist_ok=True)
        print("  restored src/moshi_local")
    if os.path.exists(os.path.join(base, "tokenizer.model")) \
            and not os.path.exists("rag/tokenizer.model"):
        shutil.copy(os.path.join(base, "tokenizer.model"), "rag/tokenizer.model")
        print("  restored rag/tokenizer.model")
    if os.path.isdir(os.path.join(base, "rag_index_dense")) \
            and not os.path.exists("rag/rag_index_dense/passages.jsonl"):
        shutil.copytree(os.path.join(base, "rag_index_dense"), "rag/rag_index_dense",
                        dirs_exist_ok=True)
        print("  restored rag/rag_index_dense")
    shutil.rmtree(tmp, ignore_errors=True)

for rel in ["src/moshi_local/moshi_distill/models/lm.py",
            "src/moshi_local/moshi_distill/models/loaders.py",
            "src/moshi_local/moshi_distill/modules/transformer.py",
            "src/moshi_local/moshi_distill/modules/sticky_latent_kv.py",
            "src/moshi_local/moshi_distill/client_utils.py",
            "src/moshi_local/moshi_distill/interleaver.py",
            "src/moshi_local/moshi_distill/rag_ref_select.py",
            "src/moshi_local/moshi_distill/ref_adapter.py",
            "src/moshi_local/moshi_distill/eval_retrieval.py",
            "src/moshi_local/moshi_distill/rag_index_v3.py",
            "src/niloys/live/rag_student_engine.py",
            "src/niloys/live/imtalker_rag_student_live.py",
            "rag/tokenizer.model",
            "rag/rag_index_dense/passages.jsonl",
            "rag/rag_index_dense/emb.npy"]:
    if not os.path.exists(rel):
        raise SystemExit(
            "MISSING: %s\nThe two RAG python modules (src/niloys/live/rag_student_engine.py, "
            "imtalker_rag_student_live.py) only exist in the UPDATED repo - push it first, "
            "then re-run from cell 4." % rel)
print("RAG integration files OK")


## 5 · Write the x86_64 build scripts into the clone

The repo's `setup_env.sh` / `prepare.sh` / `run.sh` are the **Jetson** scripts. The cells below
write the x86_64 counterparts next to them (identical to the files in this repo — the notebook
is generated from them), so the Jetson path keeps working untouched:

| file | what it changes |
|---|---|
| `requirements-x86-torch.txt` | torch 2.8.0 / torchaudio 2.8.0 / torchvision 0.23.0 / triton 3.4.0, **CUDA 12.8** builds |
| `requirements-x86-extra.txt` | `sphn==0.2.1`, `tensorrt-cu12`, `hf_transfer`, plus the **RAG deps**: `sentence-transformers`, `scikit-learn`, `tqdm`, and the upgraded `transformers 4.46.3` / `tokenizers 0.20.3` / `huggingface_hub 0.36.2` |
| `setup_env_x86.sh` | Miniforge **x86_64**, Python 3.10 env in `./env`, plus `moshi==0.2.13 --no-deps` (the STT loader) |
| `prepare_x86.sh` | the ~19 GB of avatar/base weights **plus the RAG assets**: the base7b student (~16 GB), the STT and the retriever models; GPU probe; TensorRT engine |
| `run_x86_rag.sh` | launches the **integrated RAG + avatar server** (`imtalker_rag_student_live.py`) |
| `run_watchdog_x86.sh` | restarts the server if it dies (`RUN_SCRIPT` picks the configuration) |
| `tools/build_trt_engine.py` | ONNX → TensorRT engine via the Python API |
| `tools/gpu_probe.py` | runs ATen's int4 tinygemm kernels on *this* GPU and writes `.gpu_probe.env` (used by the legacy config) |

If the repo already contains these files, the cells simply overwrite them with the same content.


In [ ]:
%%writefile requirements-x86-torch.txt
# x86_64 (RunPod / any desktop NVIDIA GPU) torch stack -- CUDA 12.8 builds.
# cu128 covers Ampere (A100/A6000), Ada (4090/L40S) and Blackwell (5090/B200, sm_120).
# Same versions as the Jetson build so the rest of requirements.txt matches exactly.
#     pip install -r requirements-x86-torch.txt
# The +cu128 local version makes the pin unambiguous even with PyPI as an extra index.
--index-url https://download.pytorch.org/whl/cu128
--extra-index-url https://pypi.org/simple
torch==2.8.0+cu128
torchaudio==2.8.0+cu128
torchvision==0.23.0+cu128
triton==3.4.0


In [ ]:
%%writefile requirements-x86-extra.txt
# x86_64 extras that on a Jetson come from JetPack or the bundled wheel:
#   sphn      -> bundled aarch64 .so in src/pylibs on the Orin; PyPI has an x86_64 wheel
#   tensorrt  -> JetPack on the Orin; pip wheel (bundles its own CUDA 12 libs) here
# Keep sphn at 0.2.x: the server needs OpusStreamReader/Writer.
sphn==0.2.1
tensorrt-cu12==10.13.3.9
# RunPod's images export HF_HUB_ENABLE_HF_TRANSFER=1, and huggingface_hub then REFUSES to
# download at all unless this package is present. It is also simply faster for the ~19 GB of
# weights, so install it rather than turning the accelerator off.
hf_transfer==0.1.9
# --- RAG configuration (run_x86_rag.sh): the RAG model's retrieval + STT ---
# These three upgrade the pins in requirements.txt (transformers 4.30.2 -> 4.46.3,
# tokenizers 0.13.3 -> 0.20.3, huggingface_hub 0.24.7 -> 0.36.2): sentence-transformers
# 3.3.1 needs transformers >= 4.41.  The avatar's Wav2Vec2 path is unaffected
# (Wav2Vec2FeatureExtractor / Wav2Vec2Model are stable across those versions).
transformers==4.46.3
tokenizers==0.20.3
huggingface_hub==0.36.2
sentence-transformers==3.3.1
scikit-learn==1.6.1
tqdm==4.68.2
# moshi 0.2.13 is installed separately with --no-deps in setup_env_x86.sh: it
# provides the CheckpointInfo the STT (kyutai/stt-1b-en_fr-candle) loads through.
# --no-deps keeps it from pulling a different sphn / huggingface_hub.



In [ ]:
%%writefile setup_env_x86.sh
#!/usr/bin/env bash
# setup_env_x86.sh -- create the Python environment for this package on an x86_64 NVIDIA GPU box
# (RunPod, Lambda, a local workstation). The Jetson counterpart is ./setup_env.sh.
#
#   ./setup_env_x86.sh
#
# Installs Miniforge (conda-forge, free) into ./.miniforge, creates a Python 3.10 env in ./env,
# installs torch 2.8.0 / torchaudio 2.8.0 / torchvision 0.23.0 / triton 3.4.0 built for CUDA 12.8
# (requirements-x86-torch.txt), then requirements.txt, then sphn + TensorRT + the RAG deps
# (sentence-transformers, moshi --no-deps, upgraded transformers/tokenizers/huggingface_hub)
# from pip (requirements-x86-extra.txt). Nothing is installed outside this folder (no apt,
# no sudo). Takes ~8-20 min (downloads ~5 GB). prepare_x86.sh, run_x86.sh and run_x86_rag.sh
# use ./env automatically. Safe to re-run: finished steps are skipped.
set -euo pipefail
HERE="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"; cd "$HERE"
MF="$HERE/.miniforge"; ENV="$HERE/env"
MF_URL="${MF_URL:-https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh}"
# keep every cache next to the package so it lands on the persistent volume, not the container disk
export CONDA_PKGS_DIRS="$HERE/.cache/conda-pkgs" PIP_CACHE_DIR="$HERE/.cache/pip" TMPDIR="$HERE/.cache/tmp"
mkdir -p "$CONDA_PKGS_DIRS" "$PIP_CACHE_DIR" "$TMPDIR"
log(){ printf '\n\033[1m==> %s\033[0m\n' "$*"; }
die(){ printf '\033[31mERROR: %s\033[0m\n' "$*" >&2; exit 1; }

log "1/5  checks"
[ "$(uname -m)" = x86_64 ] || die "this script is for x86_64; on a Jetson use ./setup_env.sh"
command -v nvidia-smi >/dev/null || die "nvidia-smi not found -- no NVIDIA driver visible in this container"
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader | sed 's/^/    /'
[ -n "$(nvidia-smi --query-gpu=name --format=csv,noheader)" ] || die "no GPU visible"

log "2/5  Miniforge -> $MF"
if [ ! -x "$MF/bin/conda" ]; then
  curl -fL --retry 3 -o "$TMPDIR/miniforge.sh" "$MF_URL" || die "could not download Miniforge from $MF_URL"
  bash "$TMPDIR/miniforge.sh" -b -p "$MF" > "$TMPDIR/miniforge_install.log" 2>&1 \
    || { tail -20 "$TMPDIR/miniforge_install.log"; die "Miniforge install failed"; }
  rm -f "$TMPDIR/miniforge.sh"
fi
"$MF/bin/conda" --version | sed 's/^/    /'

log "3/5  Python 3.10 env -> $ENV (3.10: matches the Jetson env; every pinned wheel has cp310 builds)"
if [ ! -x "$ENV/bin/python" ]; then
  "$MF/bin/conda" create -y -q -p "$ENV" python=3.10 pip > "$TMPDIR/conda_create.log" 2>&1 \
    || { tail -20 "$TMPDIR/conda_create.log"; die "conda create failed"; }
fi
PY="$ENV/bin/python"
"$PY" -c 'import sys; assert sys.version_info[:2] == (3, 10), sys.version' || die "env python is not 3.10"
"$PY" -m pip install -q --upgrade pip

log "4/5  torch/torchaudio/torchvision/triton for CUDA 12.8, then requirements.txt (~5 GB)"
if ! "$PY" -c 'import torch, sys; sys.exit(0 if torch.version.cuda and torch.cuda.is_available() else 1)' 2>/dev/null; then
  "$PY" -m pip install -r "$HERE/requirements-x86-torch.txt" \
    || die "torch install failed (index: https://download.pytorch.org/whl/cu128)"
fi
"$PY" -c 'import torch, sys; print("    torch", torch.__version__, "CUDA", torch.version.cuda, "|", torch.cuda.get_device_name(0)); sys.exit(0 if torch.cuda.is_available() else 1)' \
  || die "torch cannot see the GPU"
"$PY" -m pip install -r "$HERE/requirements.txt" || die "pip install -r requirements.txt failed"

log "5/5  sphn + TensorRT + RAG deps from pip (on a Jetson these come from src/pylibs and JetPack)"
"$PY" -m pip install -r "$HERE/requirements-x86-extra.txt" || die "sphn/TensorRT/RAG install failed"
# moshi 0.2.13 (--no-deps): provides the CheckpointInfo the STT loads through.
# --no-deps keeps it from pulling a different sphn / huggingface_hub.
"$PY" -m pip install --no-deps moshi==0.2.13 || die "moshi install failed"
# uvloop (pulled in by uvicorn[standard]) made every 80 ms step ~13 ms slower on the reference box
"$PY" -m pip uninstall -y -q uvloop 2>/dev/null || true
"$PY" - <<'T' || die "installed, but the environment check failed"
import importlib.util, torch, triton, torchaudio, torchvision, sphn, fastapi, timm, transformers, cv2
import tensorrt
import moshi, sentence_transformers, sklearn, tqdm
from moshi.models import loaders as _moshi_loaders
assert importlib.util.find_spec("uvloop") is None, "uvloop installed: remove it (pip uninstall uvloop)"
assert torch.cuda.is_available(), "torch has no CUDA"
assert hasattr(sphn, "OpusStreamReader"), "sphn too old -- need 0.2.x"
assert hasattr(_moshi_loaders, "CheckpointInfo"), "pip moshi has no CheckpointInfo (needed for the STT)"
cc = torch.cuda.get_device_capability(0)
print(f"    torch {torch.__version__} (CUDA {torch.version.cuda}) | triton {triton.__version__} | "
      f"TensorRT {tensorrt.__version__} | transformers {transformers.__version__} | "
      f"moshi {moshi.__version__ if hasattr(moshi, '__version__') else '0.2.x'} | "
      f"{torch.cuda.get_device_name(0)} sm_{cc[0]}{cc[1]} "
      f"{torch.cuda.get_device_properties(0).total_memory/2**30:.0f} GiB")
T
rm -rf "$TMPDIR"
log "ENV READY: $ENV   --  next: ./prepare_x86.sh hf_YOUR_TOKEN"


In [ ]:
%%writefile prepare_x86.sh
#!/usr/bin/env bash
# prepare_x86.sh -- weights + TensorRT engine for the PersonaPlex + IMTalker avatar on an
# x86_64 NVIDIA GPU (RunPod, Lambda, a local workstation). The Jetson counterpart is ./prepare.sh.
#
#   ./setup_env_x86.sh                             (once: Python env from scratch in ./env)
#   ./prepare_x86.sh hf_yourtokenhere              (or `hf auth login` first and omit the token)
#   PY=/path/to/python ./prepare_x86.sh hf_...     (to use your own Python 3.10 instead of ./env)
#
# Checks the environment, downloads the weights (~19 GB for the legacy full-PersonaPlex
# config, plus ~19 GB for the RAG config: the base7b student, the STT and the retrievers),
# probes which fast paths this GPU supports and builds the audio-decoder TensorRT engine
# for THIS GPU. Installs nothing. Then: ./run_x86.sh (legacy) or ./run_x86_rag.sh (RAG).
set -euo pipefail
HERE="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"; cd "$HERE"
TOKEN="${1:-${HF_TOKEN:-}}"
[ -n "$TOKEN" ] && export HF_TOKEN="$TOKEN" HUGGING_FACE_HUB_TOKEN="$TOKEN"
export HF_HOME="${HF_HOME:-$HERE/hf}"
# Python: $PY if given, else ./env from setup_env_x86.sh, else python3 on PATH
[ -z "${PY:-}" ] && [ -x "$HERE/env/bin/python" ] && PY="$HERE/env/bin/python"
PY="$(command -v "${PY:-python3}")" || { echo "python3 not found (run ./setup_env_x86.sh, or set PY=...)"; exit 1; }
W="$HERE/weights"
BASE_REPO=nvidia/personaplex-7b-v1
BASE_REV=fdaf4090a61cb315c138a1faee287ffd6c716309
OURS_REPO=niloy629/personaplex-parallel-depformer
RAG_REPO=sadmN/moshi-base7b-rag
log(){ printf '\n\033[1m==> %s\033[0m\n' "$*"; }
warn(){ printf '\033[33mWARNING: %s\033[0m\n' "$*" >&2; }
die(){ printf '\033[31mERROR: %s\033[0m\n' "$*" >&2; exit 1; }

log "1/7  environment check ($PY)"
[ "$(uname -m)" = x86_64 ] || die "this script is for x86_64; on a Jetson use ./prepare.sh"
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader | sed 's/^/    /' || die "nvidia-smi failed"
"$PY" - <<'T' || die "environment incomplete -- run ./setup_env_x86.sh (see README_RUNPOD.md)"
import sys
assert sys.version_info[:2] == (3, 10), f"need Python 3.10, got {sys.version.split()[0]}"
import torch, torchaudio, torchvision, triton, sphn, fastapi, uvicorn, timm, transformers, sentencepiece, cv2, librosa, torchdiffeq, huggingface_hub
assert torch.cuda.is_available(), "torch has no CUDA"
assert hasattr(sphn, "OpusStreamReader"), "sphn too old -- need 0.2.x"
f = torch.compile(lambda x: torch.sin(x) * 2 + 1)
x = torch.randn(1024, device="cuda"); assert torch.allclose(f(x), torch.sin(x) * 2 + 1, atol=1e-5)
print(f"    python {sys.version.split()[0]} | torch {torch.__version__} (CUDA {torch.version.cuda}) | triton {triton.__version__} | torch.compile OK")
T
echo "$PY" > "$HERE/.python_path"

# RunPod (and other) images export HF_HUB_ENABLE_HF_TRANSFER=1; huggingface_hub then refuses to
# download anything at all unless the hf_transfer package is importable. setup_env_x86.sh installs
# it (requirements-x86-extra.txt), but if this Python does not have it, turn the accelerator off
# instead of failing the download.
if [ "${HF_HUB_ENABLE_HF_TRANSFER:-0}" != "0" ] && ! "$PY" -c 'import hf_transfer' 2>/dev/null; then
  echo "    note: HF_HUB_ENABLE_HF_TRANSFER=1 but hf_transfer is not installed -> disabling it"
  export HF_HUB_ENABLE_HF_TRANSFER=0
fi

# SKIP_BASE_LM=1 skips the 16 GB full-7B model.safetensors + the legacy text tokenizer:
# the RAG config (run_x86_rag.sh) only needs the Mimi codec + the voices from this repo.
# The legacy config (run_x86.sh) needs the full download, so keep the default at 0 there.
SKIP_BASE_LM="${SKIP_BASE_LM:-0}"
if [ "$SKIP_BASE_LM" = "1" ]; then
  BASE_PATS='["tokenizer-e351c8d8-checkpoint125.safetensors", "voices.tgz"]'
  log "2/7  base Mimi + voices from $BASE_REPO (SKIP_BASE_LM=1: RAG-only, no 16 GB full-7B)"
else
  BASE_PATS='["model.safetensors", "tokenizer-e351c8d8-checkpoint125.safetensors", "tokenizer_spm_32k_3.model", "voices.tgz"]'
  log "2/7  base model from $BASE_REPO (~16 GB, gated: needs your token + accepted licence)"
fi
"$PY" - <<PYEOF || die "base download failed: is the token valid and the model licence accepted on HF?"
import os
from huggingface_hub import snapshot_download
p = snapshot_download("$BASE_REPO", revision="$BASE_REV", token=os.environ.get("HF_TOKEN"),
                      allow_patterns=$BASE_PATS)
open(os.path.join("$HERE", ".snap_path"), "w").write(p); print("    ->", p)
PYEOF
SNAP="$(cat "$HERE/.snap_path")"
mkdir -p "$W"
ln -sfn "$SNAP/tokenizer-e351c8d8-checkpoint125.safetensors" "$W/mimi.safetensors"
[ "$SKIP_BASE_LM" = "1" ] || ln -sfn "$SNAP/tokenizer_spm_32k_3.model" "$W/tokenizer_spm_32k_3.model"
[ -d "$W/voices" ] || tar xzf "$SNAP/voices.tgz" -C "$W"
[ -s "$W/voices/NATM0.pt" ] || die "voices.tgz did not contain voices/NATM0.pt"

fetch() {  # fetch <hf folder> <dest dir> <files...> : download from our public repo, sha256-check, link
  "$PY" - "$@" <<PYEOF || die "download from $OURS_REPO/\$1 failed"
import os, sys, hashlib
from huggingface_hub import hf_hub_download
folder, dest, files = sys.argv[1], sys.argv[2], sys.argv[3:]
os.makedirs(dest, exist_ok=True)
want = {}
for line in open(hf_hub_download("$OURS_REPO", f"{folder}/SHA256SUMS")):
    h, f = line.split(); want[os.path.basename(f)] = h
for f in files:
    src = hf_hub_download("$OURS_REPO", f"{folder}/{f}")
    h = hashlib.sha256()
    with open(src, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 24), b""): h.update(chunk)
    if h.hexdigest() != want[f]:
        raise SystemExit(f"hash mismatch for {f}")
    dst = os.path.join(dest, f)
    if os.path.islink(dst) or os.path.exists(dst): os.remove(dst)
    os.symlink(src, dst)
print(f"    {folder}: {len(files)} files, sha256 OK")
PYEOF
}

log "3/7  Robert_5 one-pass depformer + GAN audio decoder from $OURS_REPO (~2.9 GB, public)"
fetch robert5_gan "$W" pd_quant_onpol_robert5_plaus2.pt mimi_dec_gan_robert5.pt

log "4/7  avatar weights + Robert_5 voice + decoder ONNX from $OURS_REPO (~250 MB, public)"
fetch avatar_assets "$HERE/assets" motion_student_38M.pt renderer_narrow.pt silence_helium_mean.pt \
  blink_motion.pt ref.jpeg Robert_5.pt seanet_w12.onnx
for f in motion_student_38M.pt renderer_narrow.pt silence_helium_mean.pt blink_motion.pt ref.jpeg; do
  ln -sfn "$HERE/assets/$f" "$W/$f"
done
ln -sfn "$HERE/assets/Robert_5.pt" "$W/voices/Robert_5.pt"

log "5/7  RAG assets: base7b student from $RAG_REPO (~16 GB) + STT + retrievers (~3 GB)"
# The base7b RAG fine-tune (full 32-layer PersonaPlex backbone) is what ./run_x86_rag.sh
# serves; the tokenizer + local retrieval index ship in this repo under rag/.
RAG_DIR="$HERE/rag"
RAG_CKPT="$RAG_DIR/ckpt/rag_base7b/step_1500"
[ -s "$RAG_DIR/tokenizer.model" ] || die "rag/tokenizer.model missing -- the repo is incomplete"
[ -s "$RAG_DIR/rag_index_dense/passages.jsonl" ] || die "rag/rag_index_dense/passages.jsonl missing -- the repo is incomplete"
[ -s "$RAG_DIR/rag_index_dense/emb.npy" ] || die "rag/rag_index_dense/emb.npy missing -- the repo is incomplete"
if [ ! -s "$RAG_CKPT/student.pt" ]; then
  "$PY" - <<PYEOF || die "base7b student download from $RAG_REPO failed"
import os
from huggingface_hub import snapshot_download
p = snapshot_download("$RAG_REPO", token=os.environ.get("HF_TOKEN"),
                      allow_patterns=["ckpt/rag_base7b/step_1500/*"],
                      local_dir="$RAG_DIR")
print("    base7b student ->", os.path.join("$RAG_CKPT", "student.pt"))
PYEOF
fi
[ -s "$RAG_CKPT/student.pt" ] || die "$RAG_CKPT/student.pt still missing"
# Aux models the RAG engine downloads on first start; caching them now makes
# startup fast and the pod fully warm.  STT = transcript + VAD; MiniLM +
# cross-encoder = the local dense retriever + reranker.
"$PY" - <<PYEOF || warn "aux model pre-download failed (the server will retry on first start)"
import os
from huggingface_hub import snapshot_download
snapshot_download("kyutai/stt-1b-en_fr-candle", token=os.environ.get("HF_TOKEN"))
print("    STT cached (kyutai/stt-1b-en_fr-candle)")
snapshot_download("sentence-transformers/all-MiniLM-L6-v2")
print("    MiniLM cached")
snapshot_download("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("    cross-encoder cached")
PYEOF

log "6/7  GPU capability probe (int4 trunk kernels, VRAM, TensorRT)"
"$PY" "$HERE/tools/gpu_probe.py" "$HERE/.gpu_probe.env" || die "GPU probe failed"

log "7/7  TensorRT engine for the audio decoder (built for THIS GPU, ~1-5 min)"
# Optional: if it cannot be built the server keeps Mimi's PyTorch SEANet decoder (a few ms
# slower per step, identical audio), so a failure here is a warning, not an error.
TRTEXEC="${TRTEXEC:-/usr/src/tensorrt/bin/trtexec}"
if [ ! -s "$W/seanet_w12_fp32.engine" ]; then
  if [ -x "$TRTEXEC" ]; then
    "$TRTEXEC" --onnx="$HERE/assets/seanet_w12.onnx" --saveEngine="$W/seanet_w12_fp32.engine" \
      > "$W/trtexec_build.log" 2>&1 || { tail -20 "$W/trtexec_build.log"; rm -f "$W/seanet_w12_fp32.engine"; }
  else
    "$PY" "$HERE/tools/build_trt_engine.py" "$HERE/assets/seanet_w12.onnx" \
      "$W/seanet_w12_fp32.engine" > "$W/trtexec_build.log" 2>&1 \
      || { tail -20 "$W/trtexec_build.log"; rm -f "$W/seanet_w12_fp32.engine"; }
  fi
fi
if [ -s "$W/seanet_w12_fp32.engine" ]; then
  "$PY" - <<PYEOF || { warn "engine does not load -- removing it, the PyTorch decoder will be used"; rm -f "$W/seanet_w12_fp32.engine"; }
import tensorrt as trt
e = trt.Runtime(trt.Logger(trt.Logger.WARNING)).deserialize_cuda_engine(open("$W/seanet_w12_fp32.engine", "rb").read())
assert e is not None
print("    engine OK:", [(e.get_tensor_name(i), tuple(e.get_tensor_shape(e.get_tensor_name(i)))) for i in range(e.num_io_tensors)])
PYEOF
else
  warn "no TensorRT engine (log: weights/trtexec_build.log) -- the server will use Mimi's PyTorch SEANet decoder"
fi
log "READY  --  ./run_x86_rag.sh (RAG: base7b student + avatar)  or  ./run_x86.sh (legacy full PersonaPlex)   then open the HTTPS URL for port \${PORT:-8991}"


In [ ]:
%%writefile run_x86_rag.sh
#!/usr/bin/env bash
# run_x86_rag.sh -- PersonaPlex base7b RAG fine-tune + IMTalker real-time avatar on an
# x86_64 NVIDIA GPU (RunPod / Lambda / a local workstation).
#
# The RAG configuration of the speech2face server: the speech engine is the base7b
# RAG fine-tune (full 32-layer PersonaPlex backbone from sadmN/moshi-base7b-rag)
# served with the single-path streaming protocol (full duplex, no state machine,
# no forced silence) with STT + dense/web retrieval and the 16-token fact card
# injected into the model's own pause on <lookup>, and the avatar (motion student
# + distilled renderer + talking-head video) is the unchanged try19 pipeline.
# The legacy full-PersonaPlex configuration is ./run_x86.sh; the Jetson one is ./run.sh.
#
#   ./run_x86_rag.sh                  defaults (port 8991)
#   PORT=8080 ./run_x86_rag.sh        another port (not 8998: refused by the server)
#   RETRIEVER=local ./run_x86_rag.sh  local index only (fully offline, no web search)
#   RETRIEVER=web ./run_x86_rag.sh    web search only (drops the local index models)
set -u
HERE="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
SRC="$HERE/src"; W="$HERE/weights"; RAG="$HERE/rag"
[ -s "$HERE/.snap_path" ] && [ -s "$HERE/.python_path" ] || { echo "run ./prepare_x86.sh <hf_token> first"; exit 1; }
PY="${PY:-$(cat "$HERE/.python_path")}"   # the Python prepare_x86.sh checked
SNAP="$(cat "$HERE/.snap_path")"
PORT="${PORT:-8991}"   # 8998 is refused by the server (reserved for a legacy production server)
export HF_HOME="${HF_HOME:-$HERE/hf}"

# RunPod (and other) images export HF_HUB_ENABLE_HF_TRANSFER=1; huggingface_hub then refuses to
# download anything at all unless the hf_transfer package is importable. setup_env_x86.sh installs
# it (requirements-x86-extra.txt), but if this Python does not have it, turn the accelerator off
# instead of failing the download.
if [ "${HF_HUB_ENABLE_HF_TRANSFER:-0}" != "0" ] && ! "$PY" -c 'import hf_transfer' 2>/dev/null; then
  echo "    note: HF_HUB_ENABLE_HF_TRANSFER=1 but hf_transfer is not installed -> disabling it"
  export HF_HUB_ENABLE_HF_TRANSFER=0
fi
# compiled-kernel caches next to the package so they survive on the persistent volume
export TRITON_CACHE_DIR="${TRITON_CACHE_DIR:-$HERE/.cache/triton}" TORCHINDUCTOR_CACHE_DIR="${TORCHINDUCTOR_CACHE_DIR:-$HERE/.cache/inductor}"
export DESIGN_A_DIR="$SRC/niloys" IMTALKER_DIR="$SRC/imtalker"
# PYTHONPATH: src (for the moshi_local package) + src/niloys (live modules).
# NOTE: src/moshi_pkg is deliberately NOT on the path -- the STT is loaded
# through the pip moshi>=0.2.x package, and the RAG engine only uses
# moshi_local.moshi_distill.  src/pylibs stays off too (aarch64 sphn).
export PYTHONPATH="$SRC:$SRC/niloys:${PYTHONPATH:-}"
unset TRY19_DEC_CKPT
export NATIVE_PASSTHROUGH=1
export TRY19_STUDENT_GRAPH="${TRY19_STUDENT_GRAPH:-1}" TRY19_RENDER_FP16="${TRY19_RENDER_FP16:-1}"
export TRY19_CODE_GUARD="${TRY19_CODE_GUARD:-1}"
# torch>=2.6 loads with weights_only=True by default; the RAG student checkpoint
# carries plain python metadata, so keep the old behaviour (we only load our own files).
export TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

# RAG assets (downloaded / verified by prepare_x86.sh; tokenizer + index ship in the repo)
export RAG_STUDENT_CKPT="${RAG_STUDENT_CKPT:-$RAG/ckpt/rag_base7b/step_1500}"
export RAG_INDEX="${RAG_INDEX:-$RAG/rag_index_dense}"
export RAG_TOKENIZER="${RAG_TOKENIZER:-$RAG/tokenizer.model}"
export RETRIEVER="${RETRIEVER:-hybrid}" WEB_PROVIDER="${WEB_PROVIDER:-tavily}"
[ -s "$RAG_STUDENT_CKPT/student.pt" ] || { echo "missing $RAG_STUDENT_CKPT/student.pt -- re-run ./prepare_x86.sh <hf_token>"; exit 1; }
[ -s "$RAG_TOKENIZER" ] || { echo "missing $RAG_TOKENIZER -- the repo is incomplete (rag/ must contain tokenizer.model)"; exit 1; }
[ -s "$RAG_INDEX/passages.jsonl" ] || { echo "missing $RAG_INDEX/passages.jsonl -- the repo is incomplete (rag/ must contain rag_index_dense/)"; exit 1; }

mkdir -p "$SRC/niloys/live/static" "$HERE/captures"
ln -sfn "$SRC/imtalker/static/assets" "$SRC/niloys/live/static/assets"
ln -sfn "$SRC/niloys/static/index_try21_no_idle_dissolve.html" "$SRC/niloys/live/static/index.html"
exec "$PY" -u "$SRC/niloys/live/imtalker_rag_student_live.py" \
  --live_module imtalker_native_passthrough \
  --student_ckpt "$W/motion_student_38M.pt" --distilled_renderer "$W/renderer_narrow.pt" \
  --motion_gain "${GAIN:-1.2}" \
  --blend_frames 0 --jump_thresh 0.15 --mouth_close 0.5 \
  --host 0.0.0.0 --port "$PORT" \
  --html_path "$SRC/niloys/static/index_try21_no_idle_dissolve.html" \
  --generator_path "$W/generator_last.ckpt" --renderer_path "$W/renderer.ckpt" \
  --adapter_path "$W/adapter_last.pt" \
  --adapter_type unitalk_last_layer --adapter_num_layers 12 --adapter_dropout 0.0 \
  --adapter_window_mode lookahead --adapter_future_steps 0 \
  --ref_path "$W/ref.jpeg" --wav2vec_model_path "$W/wav2vec2-base-960h" \
  --moshi_root "$W" --mimi_hf_repo nvidia/personaplex-7b-v1 \
  --mimi_weight "$W/mimi.safetensors" \
  --tokenizer "$RAG_TOKENIZER" \
  --text_prompt "${TEXT_PROMPT:-You are Robert, an assistant from RB Labs. You have knowledge about crypto, finance, investment, technology, and general topics. Answer every question completely, descriptively, and with useful detail.}" \
  --voice_prompt "${VOICE:-Robert_5.pt}" --voice_prompt_dir "$W/voices" \
  --enable_moshi_reply --direct_reply_hidden --reply_hidden_steps_per_chunk 1 \
  --audio_chunk_sec 2.0 --wav2vec_sec 2.0 --fm_chunk_frames 2 --helium_deque_size 25 \
  --prebuffer_chunks "${PREBUFFER:-3}" --render_sub_batch 2 --renderer_precision fp32 \
  --frame_q_backpressure 3 --buffer_ms 80 --skip_fm_audio_encoder \
  --assistant_speech_rms_threshold 0.006 --assistant_speech_hold_chunks 1 \
  --motion_ref_blend 0.0 --motion_prior_noise_blend 0.0 \
  --a_cfg_scale 1.24 --nfe 3 --seed 42 --noise_seed 42 --shared_noise \
  --fp32 --tf32 \
  --silence_helium_path "$W/silence_helium_mean.pt" \
  --jpeg_quality 90 --device cuda --reply_audio_gain 1.0 --output_audio_codec opus \
  --capture_dir "$HERE/captures" \
  --blink_motion_path "$W/blink_motion.pt"

echo "x86-rag: base7b RAG (streaming) + retriever=$RETRIEVER/$WEB_PROVIDER + try19 avatar voice=${VOICE:-Robert_5.pt} port=$PORT"


In [ ]:
%%writefile run_watchdog_x86.sh
#!/usr/bin/env bash
# run_watchdog_x86.sh -- keep the x86 server up: restart it if it exits or hits a CUDA
# device-side assert.  RUN_SCRIPT picks the configuration: ./run_x86.sh (legacy full
# PersonaPlex, the default) or ./run_x86_rag.sh (base7b RAG + avatar).
#   RUN_SCRIPT=run_x86_rag.sh ./run_watchdog_x86.sh &
# Log: run.log (current), crashlogs/ (saved logs of crashed runs).
# Stop: pkill -f run_watchdog_x86.sh; pkill -f speech2face/src
HERE="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"; cd "$HERE"; mkdir -p crashlogs
RUN_SCRIPT="${RUN_SCRIPT:-run_x86.sh}"
exec 9>"$HERE/.watchdog_x86.lock"; flock -n 9 || { echo "watchdog already running"; exit 1; }
while true; do
  ./"$RUN_SCRIPT" > run.log 2>&1 &
  PID=$!; echo "[$(date +%F_%T)] started pid $PID ($RUN_SCRIPT)" >> crashlogs/watchdog.log
  sleep 60
  while kill -0 $PID 2>/dev/null; do
    if grep -q "device-side assert" run.log; then
      cp run.log crashlogs/crash_$(date +%m%d_%H%M%S).log
      echo "[$(date +%F_%T)] device-side assert -> restarting" >> crashlogs/watchdog.log
      kill -9 $PID; break
    fi
    sleep 5
  done
  wait $PID 2>/dev/null; sleep 5
done


In [ ]:
%%writefile tools/build_trt_engine.py
"""tools/build_trt_engine.py -- build a TensorRT engine from an ONNX file with the Python API.

    python tools/build_trt_engine.py <model.onnx> <out.engine> [workspace_GiB]

Same job as `trtexec --onnx=... --saveEngine=...`, for machines where trtexec is not installed
(the pip TensorRT wheels ship the libraries and the Python bindings but no trtexec binary;
JetPack ships trtexec at /usr/src/tensorrt/bin/trtexec). Engines are tied to the GPU and the
TensorRT version, so this always runs on the machine that will serve.

The SEANet window decoder used here has fully static shapes, so no optimisation profile is
needed; precision stays fp32 to match weights/seanet_w12_fp32.engine on the Orin.
"""
import os
import sys

import tensorrt as trt


def build(onnx_path: str, engine_path: str, workspace_gib: float = 4.0) -> None:
    logger = trt.Logger(trt.Logger.WARNING)
    builder = trt.Builder(logger)
    network = builder.create_network()
    parser = trt.OnnxParser(network, logger)
    with open(onnx_path, "rb") as f:
        if not parser.parse(f.read()):
            for i in range(parser.num_errors):
                print(f"onnx parse error: {parser.get_error(i)}", file=sys.stderr)
            raise SystemExit(f"could not parse {onnx_path}")

    cfg = builder.create_builder_config()
    cfg.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, int(workspace_gib * (1 << 30)))
    print(f"[build_trt_engine] TensorRT {trt.__version__}: {onnx_path} -> {engine_path} "
          f"(fp32, workspace {workspace_gib:g} GiB)", flush=True)
    for i in range(network.num_inputs):
        t = network.get_input(i)
        print(f"    input  {t.name} {tuple(t.shape)} {t.dtype}")
    for i in range(network.num_outputs):
        t = network.get_output(i)
        print(f"    output {t.name} {tuple(t.shape)} {t.dtype}")

    plan = builder.build_serialized_network(network, cfg)
    if plan is None:
        raise SystemExit("TensorRT build_serialized_network returned None")
    tmp = engine_path + ".tmp"
    with open(tmp, "wb") as f:
        f.write(plan)
    os.replace(tmp, engine_path)
    print(f"[build_trt_engine] wrote {engine_path} ({os.path.getsize(engine_path)/2**20:.1f} MiB)",
          flush=True)


if __name__ == "__main__":
    if len(sys.argv) < 3:
        raise SystemExit(__doc__)
    build(sys.argv[1], sys.argv[2], float(sys.argv[3]) if len(sys.argv) > 3 else 4.0)


In [ ]:
%%writefile tools/gpu_probe.py
"""tools/gpu_probe.py -- report the GPU and decide which fast paths this GPU can actually run.

    python tools/gpu_probe.py [out.env]

The Orin configuration quantises the 7B trunk to int4 with ATen's tinygemm kernels
(torch.ops.aten._weight_int4pack_mm). That kernel is not compiled for every CUDA
architecture, so on some cards it raises instead of running. This script tries it with the
exact code path the server uses (src/niloys/int4_linear.py) and writes a shell-sourceable file:

    TRY19_INT4_G=32          # int4 trunk works -> use it (needs ~14 GB of VRAM)
    TRY19_INT4_G=0           # int4 unavailable  -> bf16 trunk (needs ~26 GB of VRAM)
    TRY19_DEP_GATING_INT4=1/0

run_x86.sh sources that file automatically if it is next to the package as .gpu_probe.env.
"""
import os
import sys

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
sys.path.insert(0, os.path.join(ROOT, "src", "niloys"))

import torch  # noqa: E402

OUT = sys.argv[1] if len(sys.argv) > 1 else os.path.join(ROOT, ".gpu_probe.env")


def main() -> int:
    if not torch.cuda.is_available():
        print("ERROR: torch cannot see a CUDA GPU", file=sys.stderr)
        return 1
    props = torch.cuda.get_device_properties(0)
    cc = (props.major, props.minor)
    vram = props.total_memory / 2 ** 30
    print(f"GPU            : {props.name}  sm_{cc[0]}{cc[1]}  {vram:.1f} GiB")
    print(f"torch          : {torch.__version__} (CUDA {torch.version.cuda})")
    try:
        import tensorrt
        print(f"TensorRT       : {tensorrt.__version__}")
    except Exception as e:
        print(f"TensorRT       : NOT AVAILABLE ({type(e).__name__}) -- the SEANet decoder "
              "stays in PyTorch (a few ms slower, same audio)")

    # bf16 is required: the trunk weights are bf16 and the int4 kernels take bf16 activations.
    if cc[0] < 8:
        print("bf16           : NOT SUPPORTED on sm_%d%d -- this pipeline needs Ampere or newer"
              % cc, file=sys.stderr)
        return 1
    print("bf16           : OK")

    int4_ok, int4_err = False, ""
    try:
        from int4_linear import Int4Linear, group_quantize
        lin = torch.nn.Linear(512, 256, bias=False, device="cuda", dtype=torch.bfloat16)
        packed, sz = group_quantize(lin.weight.data, 32)
        q = Int4Linear(packed, sz, None, 256, 32)
        x = torch.randn(4, 512, device="cuda", dtype=torch.bfloat16)
        ref, got = lin(x).float(), q(x).float()
        cos = torch.nn.functional.cosine_similarity(ref.flatten(), got.flatten(), 0).item()
        if not torch.isfinite(got).all():
            raise RuntimeError("int4 matmul produced non-finite values")
        if cos < 0.9:
            raise RuntimeError(f"int4 matmul cosine {cos:.3f} vs bf16 -- kernel misbehaving")
        int4_ok = True
        print(f"int4 tinygemm  : OK (cosine {cos:.4f} vs bf16)")
    except Exception as e:                                   # noqa: BLE001
        int4_err = f"{type(e).__name__}: {e}"
        print(f"int4 tinygemm  : UNAVAILABLE -- {int4_err}")

    try:
        f = torch.compile(lambda t: torch.sin(t) * 2 + 1)
        t = torch.randn(1024, device="cuda")
        assert torch.allclose(f(t), torch.sin(t) * 2 + 1, atol=1e-5)
        print("torch.compile  : OK")
    except Exception as e:                                   # noqa: BLE001
        print(f"torch.compile  : FAILED ({type(e).__name__}: {e})")

    need = 14 if int4_ok else 26
    print(f"\nplan           : {'int4 G=32 trunk' if int4_ok else 'bf16 trunk (int4 kernels unusable)'}"
          f", needs ~{need} GB of VRAM, this GPU has {vram:.0f} GB")
    if vram < need:
        print(f"WARNING        : only {vram:.0f} GB of VRAM -- expect CUDA OOM. Use a bigger GPU "
              f"(>= {need} GB).", file=sys.stderr)

    with open(OUT, "w") as fh:
        fh.write("# written by tools/gpu_probe.py -- sourced by run_x86.sh\n")
        fh.write(f"TRY19_INT4_G={32 if int4_ok else 0}\n")
        fh.write(f"TRY19_DEP_GATING_INT4={1 if int4_ok else 0}\n")
    print(f"wrote          : {OUT}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
sh("chmod +x setup_env_x86.sh prepare_x86.sh run_x86.sh run_x86_rag.sh run_watchdog_x86.sh "
   "setup_env.sh prepare.sh run.sh run_watchdog.sh 2>/dev/null; "
   "ls -l *_x86.sh requirements-x86-*.txt tools/build_trt_engine.py tools/gpu_probe.py")


## 6 · Build the Python environment  *(~10–25 min, ~6 GB)*

Miniforge → `./.miniforge`, a Python 3.10 env → `./env`, then torch (CUDA 12.8), the pinned
`requirements.txt`, `sphn` + TensorRT + the **RAG deps** (sentence-transformers, upgraded
transformers/tokenizers/huggingface_hub) from pip, and `moshi==0.2.13 --no-deps` for the STT.

Nothing is installed outside the clone — no `pip install` into the base image. Safe to re-run:
finished steps are skipped. The last line must be `ENV READY`.


In [ ]:
t0 = time.time()
sh("./setup_env_x86.sh")
print("\n--- setup_env_x86.sh took %.1f min ---" % ((time.time() - t0) / 60))


### Sync the pip extras

Cheap and idempotent (a few seconds once the env exists). Run it if you built `./env` from an
older copy of this notebook — in particular it installs `hf_transfer`, which RunPod's images
*require* for any Hugging Face download because they export `HF_HUB_ENABLE_HF_TRANSFER=1`, and
the RAG deps (sentence-transformers, moshi) if they are missing.


In [ ]:
sh("./env/bin/python -m pip install -r requirements-x86-extra.txt")
sh("./env/bin/python -m pip install --no-deps moshi==0.2.13")
sh("./env/bin/python -c 'import sphn, tensorrt, hf_transfer, huggingface_hub, transformers; "
   "import moshi, sentence_transformers; "
   "print(\"sphn/tensorrt/hf_transfer | tensorrt\", tensorrt.__version__, "
   "\"| hub\", huggingface_hub.__version__, \"| tf\", transformers.__version__)'")


## 7 · Download the weights and build the TensorRT engine  *(~15–40 min, ~23 GB)*

1. environment check
2. `nvidia/personaplex-7b-v1` — **Mimi codec + voices only** (~1 GB, **gated**). With
   `SKIP_BASE_LM=1` (the default) the 16 GB full-7B `model.safetensors` is **not** downloaded —
   the RAG student does not use it. Set `SKIP_BASE_LM=0` in the config cell if you also want the
   legacy full-PersonaPlex config.
3. the Robert_5 one-pass depformer + GAN Mimi decoder (~2.9 GB, public, sha256-checked) — *legacy config*
4. avatar weights, the Robert_5 voice and the decoder ONNX (~250 MB, public, sha256-checked)
5. **the RAG assets**: the base7b student (~16 GB, `sadmN/moshi-base7b-rag`), the STT
   (`kyutai/stt-1b-en_fr-candle`, ~2.4 GB) and the retriever models (MiniLM + cross-encoder)
6. the GPU capability probe
7. the TensorRT engine for the audio decoder, built for **this** GPU (legacy config; optional)

Step 2 fails if the token is wrong or the account has not accepted the model licence.
Safe to re-run; it skips what is already there. The last line must be `READY`.


In [ ]:
t0 = time.time()
sh("./prepare_x86.sh " + HF_TOKEN)
print("\n--- prepare_x86.sh took %.1f min ---" % ((time.time() - t0) / 60))


### What the probe decided + the RAG assets

`TRY19_INT4_G` only matters for the **legacy** full-PersonaPlex config (`run_x86.sh`); the RAG
model is served in bf16 regardless. This cell also confirms the base7b student and the local
retrieval index are in place.


In [ ]:
sh("cat .gpu_probe.env 2>/dev/null; echo; "
   "ls -lL weights/seanet_w12_fp32.engine 2>/dev/null || echo 'no TensorRT engine "
   "(the PyTorch SEANet decoder will be used - only matters for the legacy config)'; echo; "
   "ls -lh rag/ckpt/rag_base7b/step_1500/student.pt rag/tokenizer.model "
   "rag/rag_index_dense/passages.jsonl rag/rag_index_dense/emb.npy", check=False)


## 8 · Start the RAG + avatar server

It runs detached with `setsid`, so it keeps serving while you use the rest of the notebook and
survives a kernel restart. Output goes to `run.log`. `RETRIEVER` comes from the config cell
(`hybrid` = web search + local index fallback; `local` = fully offline).


In [ ]:
sh("pkill -f run_watchdog_x86.sh; pkill -f imtalker_rag_student_live.py; sleep 2", check=False)
sh("PORT=%d VOICE=%s TEXT_PROMPT=%s RETRIEVER=%s nohup setsid ./run_x86_rag.sh > run.log 2>&1 < /dev/null & "
   "sleep 3; echo started:; pgrep -af imtalker_rag_student_live.py | head -3"
   % (PORT, VOICE, json.dumps(TEXT_PROMPT), RETRIEVER))


### Wait until it has loaded

Loading the base7b student, the STT, the retriever models, capturing the CUDA graphs and compiling
the Triton kernels takes a few minutes the first time (the kernel cache in `.cache/` makes later
starts quicker). This cell polls `/health` until `loaded: true`.


In [ ]:
def wait_loaded(port, timeout=1800):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            with urllib.request.urlopen("http://127.0.0.1:%d/health" % port, timeout=5) as r:
                h = json.loads(r.read())
            if h.get("loaded"):
                print("LOADED after %.0fs" % (time.time() - t0))
                return True
        except Exception:
            pass
        if subprocess.run(["bash", "-c", "pgrep -f imtalker_rag_student_live.py"],
                          capture_output=True).returncode != 0:
            print("\n!! server process died - last log lines:")
            sh("tail -60 run.log", check=False)
            return False
        tail = subprocess.run(["bash", "-c", "tail -1 run.log"], capture_output=True,
                              text=True, cwd=REPO_DIR).stdout.strip()
        print("  ... %5.0fs  %s" % (time.time() - t0, tail[:110]), flush=True)
        time.sleep(15)
    print("\n!! timed out")
    sh("tail -40 run.log", check=False)
    return False

ok = wait_loaded(PORT)
print()
# the feature lines that must be in the log
sh("grep -aE 'x86-rag:|\\[rag\\]|\\[try-rag\\]|Uvicorn running' run.log | tail -30 "
   "|| echo '(none found yet)'", check=False)


## 9 · Open it in Chrome

**The microphone only works on `https://` or `http://localhost`.** Plain `http://<pod-ip>:8991`
loads the page, but Chrome refuses the microphone on it. Two ways to get HTTPS:

### Option A — the RunPod HTTP proxy  *(use this if 8991 is in the pod's HTTP ports)*


In [ ]:
pod = os.environ.get("RUNPOD_POD_ID", "")
if pod:
    print("  https://%s-%d.proxy.runpod.net/\n" % (pod, PORT))
    print("If that 502s, port %d is not in this pod's HTTP ports - either add it (a template" % PORT)
    print("change needs the pod recreated) or use the Cloudflare tunnel below.")
else:
    print("RUNPOD_POD_ID is not set - this is not a RunPod pod, or it is an older image.")
    print("Use the Cloudflare tunnel below, or an SSH tunnel:")
    print("  ssh -L %d:localhost:%d root@<pod-host> -p <ssh-port>   then http://localhost:%d/"
          % (PORT, PORT, PORT))


### Option B — Cloudflare quick tunnel  *(no pod configuration, no account)*

Downloads `cloudflared` and prints a `https://<random>.trycloudflare.com` URL that proxies to the
server, websockets included. Leave the tunnel running for as long as you want the URL; re-run the
cell to get a fresh one.


In [ ]:
import re
os.makedirs(".bin", exist_ok=True)
CF = os.path.abspath(".bin/cloudflared")
if not os.path.exists(CF):
    print("downloading cloudflared ...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CF)
    os.chmod(CF, 0o755)

sh("pkill -f 'cloudflared tunnel'; sleep 1", check=False)
tun = subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--url", "http://localhost:%d" % PORT],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
public, t0 = None, time.time()
while time.time() - t0 < 90:
    line = tun.stdout.readline()
    if not line:
        break
    m = re.search(r"https://[-\w.]+\.trycloudflare\.com", line)
    if m:
        public = m.group(0)
        break
if public:
    print("\n" + "=" * 72)
    print("  OPEN THIS IN CHROME:   %s/" % public)
    print("=" * 72)
    print("\nClick 'Start conversation', allow the microphone, and speak.")
    print("Ask a factual question (e.g. 'what year was the Eiffel tower completed?') -")
    print("the model retrieves a passage and answers from it (watch run.log for '[rag]').")
else:
    print("could not get a tunnel URL; last output:")
    print(tun.stdout.read()[-2000:] if tun.stdout else "")


## 10 · Headless end-to-end check (no browser, no microphone)

Streams 20 s of recorded speech into the server, then silence, and reports what came back.
`PASS` = it heard the question, answered with speech and streamed video above 15 fps. The median
step time must stay under **80 ms** — that is the whole budget for one 80 ms audio frame (the
RAG step also runs the STT, so watch this number on smaller GPUs). The answer differs every run.


In [ ]:
sh("$(cat .python_path) tools/test_session.py %d 40" % PORT, check=False)


## 11 · Day-to-day: logs, restart, stop


In [ ]:
sh("tail -60 run.log", check=False)


In [ ]:
# RAG turns, retrievals, step times and anything worth knowing about
sh("grep -aE '\\[rag\\]|step time|over budget|WARNING|Error|Traceback' run.log | tail -50", check=False)


In [ ]:
# restart under the watchdog instead (auto-restarts on a crash; logs to run.log + crashlogs/)
sh("pkill -f imtalker_rag_student_live.py; sleep 2; "
   "PORT=%d RETRIEVER=%s RUN_SCRIPT=run_x86_rag.sh nohup setsid ./run_watchdog_x86.sh > /dev/null 2>&1 < /dev/null & "
   "sleep 3; pgrep -af run_watchdog_x86.sh" % (PORT, RETRIEVER), check=False)


In [ ]:
# stop everything
sh("pkill -f run_watchdog_x86.sh; pkill -f imtalker_rag_student_live.py; "
   "pkill -f 'cloudflared tunnel'; sleep 2; "
   "nvidia-smi --query-gpu=memory.used --format=csv,noheader", check=False)


## 12 · Tuning and troubleshooting

### Knobs — set them in the config cell (or export them and re-run `./run_x86_rag.sh`)

| variable | default | effect |
|---|---|---|
| `PORT` | 8991 | server port (8998 is refused by the server) |
| `VOICE` | `Robert_5.pt` | voice prompt from `weights/voices/` (the avatar's motion student was trained on Robert_5 speech) |
| `TEXT_PROMPT` | Robert, RB Labs | persona / system prompt |
| `RETRIEVER` | `hybrid` | passage source: `hybrid` (web + local fallback), `local` (offline), `web` (no local index models) |
| `WEB_PROVIDER` | `tavily` | web search provider (`tavily` / `brave` / `ddg`); `TAVILY_API_KEY` / `BRAVE_API_KEY` override the built-in key |
| `WEB_WAIT_FRAMES` | 8 | pause frames the model spends waiting for an in-flight web search (80 ms each) |
| `VAD_THRESHOLD` | 0.5 | STT VAD score that counts as "user stopped talking" |
| `PREBUFFER` | 3 | pre-roll chunks of 80 ms. Raise it if audio stutters on a slow network |
| `TRY19_RENDER_FP16` | 1 | fp16 renderer. `0` for fp32 (slower, 64 dB difference) |
| `GAIN` | 1.2 | motion gain of the avatar |

Example: `PORT=8080 RETRIEVER=local ./run_x86_rag.sh 2>&1 | tee run.log`

### Troubleshooting

| symptom | fix |
|---|---|
| microphone blocked, no "allow" prompt | the page must be on `https://` or `http://localhost`. Use the Cloudflare tunnel or an SSH tunnel |
| the proxy URL 502s | port `8991` is not in the pod's HTTP ports, or the server has not finished loading — check `/health` first |
| `MISSING: src/niloys/live/rag_student_engine.py` in cell 4b | the clone is the **old** repo — push the updated one, then re-run from cell 4 |
| `missing rag/ckpt/rag_base7b/step_1500/student.pt` at launch | re-run `./prepare_x86.sh <hf_token>` (step 7) — the base7b student download failed or was skipped |
| `401/403` on `nvidia/personaplex-7b-v1` | gated repo: accept the licence on its model page with the same account as your `HF_TOKEN`, then re-run step 7 |
| `the imported moshi package has no CheckpointInfo` | `src/moshi_pkg` leaked onto `PYTHONPATH` — launch only through `run_x86_rag.sh` (it keeps the path clean) |
| `RAG tokenizer must have 32003 ids` | `--tokenizer` points at the old 32000-id file — it must be `rag/tokenizer.model` (run_x86_rag.sh sets it) |
| no retrieval in the log (`[rag] transcript=` never appears) | the STT did not fire: speak a full question and pause; check `VAD_THRESHOLD`; small talk (hello/thanks) is deliberately not retrieved on |
| web search always fails | no egress to the search API, or the built-in Tavily key is exhausted — set `TAVILY_API_KEY`, or `RETRIEVER=local` |
| `CUDA out of memory` while loading | something else holds VRAM (`nvidia-smi`), or the card has < ~25 GB free — close other processes, or `RETRIEVER=web` to drop the local index models |
| step time over 80 ms, choppy audio | `env/bin/pip uninstall uvloop`; make sure nothing else uses the GPU; one server per GPU. The STT adds per-frame work — on a small GPU it can push the step over budget |
| browser connects then disconnects at once | `sphn` must be the **pip** 0.2.x build. `src/pylibs` holds an aarch64 `sphn` and must stay off `PYTHONPATH` — `run_x86_rag.sh` already keeps it off |
| `ImportError: libGL.so.1` | re-run cell 3 |
| disk full mid-download | the volume is under ~75 GB. `du -sh hf weights env rag .cache` shows where it went |

### Known behaviour

* The interaction is **full duplex** (the streaming RAG protocol, like the legacy passthrough):
  the model hears you while it speaks and you can talk over it — there is no turn-taking state
  machine and no barge-in reset, so long conversations keep their context (the KV cache persists
  for the whole session).
* The card budget is 16 tokens, so answers are best for single-fact questions; with very long
  continuous speech the avatar video can fall a few seconds behind.
* The motion student was trained on the **full** PersonaPlex's hidden states — and the base7b
  RAG model IS the full 32-layer PersonaPlex (fine-tuned), so its hidden states match the motion
  student's training distribution (better lip-sync than a pruned backbone). The context window is
  ~4 minutes; reconnecting starts a fresh session.
* The legacy full-PersonaPlex configuration is untouched: `./run_x86.sh` (see README_RUNPOD.md).
* Everything lives under `/workspace/speech2face`. On a pod restart with the same volume, skip
  cells 6 and 7 and go straight to cell 8.
